# Lab 3B: Fine-tune Llama 3.2 3B with Experiment Tracking

## Overview

In this lab, you fine-tune Llama 3.2 3B for text summarization using the
**SageMaker Python SDK v3**. The workflow uses `JumpStartConfig` as the model
contract, `ModelTrainer` for managed fine-tuning, `ModelBuilder` for both base
and fine-tuned deployment, and typed `Endpoint.invoke()` for inference.

The MLflow run records the dataset lineage, training job, model artifact,
inference image, environment, and endpoint names consumed by Lab 3C.


## Step 1: Setup and Install Dependencies

First, you'll install the required libraries and initialize our SageMaker session. This establishes the execution context for our training job.

<div style="padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404;">
<strong>⚠️ Important:</strong> The cell below installs libraries and restarts the kernel. After the restart, continue with the next cell.
</div>

In [1]:
import importlib.metadata as metadata
from packaging.version import Version
import datasets

sdk_version = Version(metadata.version("sagemaker"))
datasets_version = Version(datasets.__version__)
print(f"SageMaker SDK version: {sdk_version}")
print(f"datasets version:      {datasets_version}")

assert Version("3") <= sdk_version < Version("4"), (
    f"This notebook requires sagemaker>=3,<4; found {sdk_version}. "
    "Run the installation cell and wait for the kernel restart."
)
assert datasets_version >= Version("4.4.1"), (
    f"This notebook requires datasets>=4.4.1; found {datasets_version}."
)
print("✓ Dependency versions are compatible")


SageMaker SDK version: 3.27.0
datasets version:      5.1.0
✓ Dependency versions are compatible


In [2]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import boto3
import mlflow
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.jumpstart.configs import JumpStartConfig
from sagemaker.core.training.configs import Compute, InputData, OutputDataConfig
from sagemaker.core.resources import Endpoint
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.serve.model_builder import ModelBuilder

# Shared loader at the repo root. Imported HERE rather than further down,
# because the execution role below comes from the .env it reads -- the MLflow
# cell in Step 3 also imports it, which is why this used to raise NameError
# when the role lookup was moved up to this cell.
for _candidate in [os.getcwd(), *[str(p) for p in Path(os.getcwd()).parents]]:
    if os.path.exists(os.path.join(_candidate, "workshop_env.py")):
        sys.path.insert(0, _candidate)
        break
from workshop_env import load_workshop_env

# SDK v3 session.
sess = Session()
region = sess.boto_region_name
bucket = sess.default_bucket()  # Same contract as Lab 2A.

# Execution role from the shared .env that lab0-setup discovered from the
# deployed domain, so Lab 2A and Lab 3B agree on one value. Side effect worth
# knowing: load_workshop_env() also loads every other .env key into os.environ.
# get_execution_role() stays as a fallback -- it needs a SageMaker execution
# context and raises on a local kernel.
env = load_workshop_env()
role = env.get("SAGEMAKER_EXEC_ROLE", "")
role_source = ".env (SAGEMAKER_EXEC_ROLE)"
if not role:
    try:
        role = get_execution_role()
        role_source = "get_execution_role()"
    except Exception as exc:
        raise RuntimeError(
            "Could not resolve a SageMaker execution role. SAGEMAKER_EXEC_ROLE is "
            "empty in .env and get_execution_role() failed "
            f"({type(exc).__name__}: {exc}). Run lab0-setup/setup.ipynb to "
            "populate .env. Unlike Lab 2A, this lab cannot continue without a "
            "role: it creates training jobs and endpoints with it."
        ) from exc

if not role.startswith("arn:aws:iam::"):
    raise ValueError(f"SAGEMAKER_EXEC_ROLE is not a role ARN: {role!r}")

account_id = boto3.client("sts", region_name=region).get_caller_identity()["Account"]
sm_client = boto3.client("sagemaker", region_name=region)
s3_client = boto3.client("s3", region_name=region)

print(f"Amazon SageMaker role: {role}")
print(f"  resolved from:       {role_source}")
print(f"Account ID:            {account_id}")
print(f"Amazon S3 bucket:      {bucket}")
print(f"AWS Region:            {region}")

INFO:sagemaker.config:Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
INFO:sagemaker.config:Not applying SDK defaults from location: /Users/rvvittal/Library/Application Support/sagemaker/config.yaml
INFO:botocore.tokens:Loading cached SSO token for sso-main


Amazon SageMaker role: arn:aws:iam::716664005094:role/smai-genai-ml-std-product-role-
  resolved from:       .env (SAGEMAKER_EXEC_ROLE)
Account ID:            716664005094
Amazon S3 bucket:      sagemaker-us-east-1-716664005094
AWS Region:            us-east-1


## Step 2: Deploy the Base Model with SDK v3


Next, you will deploy the base Llama 3.2 3B model so that you can later compare its performance with the fine-tuned model for your summarization use case

In [3]:
# Pin the existing 1.x model contract. The current SDK v3 Hub API rejects the
# old wildcard "1.*", and JumpStart 2.0.0 may have different input/output
# signatures. Pinning 1.1.9 keeps this lab reproducible.
model_id = "meta-textgeneration-llama-3-2-3b"
model_version = "1.1.9"
inference_instance_type = "ml.g6.2xlarge"

jumpstart_config = JumpStartConfig(
    model_id=model_id,
    model_version=model_version,
    accept_eula=True,
)


<div style="background-color: #d4edda; border: 1px solid #c3e6cb; border-radius: 4px; padding: 12px; margin: 10px 0;">
<b>✓ Llama Model EULA Acceptance</b><br>
To deploy Llama models using SageMaker JumpStart, you must accept Meta's End User License Agreement (EULA). In the notebook, set <code>accept_eula=true</code> in the estimator configuration. By doing so, you acknowledge that you have read and agree to the terms of the EULA, available at https://ai.meta.com/resources/models-and-libraries/llama-downloads/. Deployment will fail if this parameter is not set to true.
</div>


> **⏱️ Note:** The deployment job will take approximately 10 minutes to complete.


In [4]:
# --- Capacity-aware deployment helper --------------------------------------
# InsufficientInstanceCapacity means AWS has no free instances of that type in
# the Region right now - it is not a config problem. This helper walks a
# prioritized list of GPU instance types (all supported by this 3B model, all
# 24 GB) and falls back to the next one on capacity errors. Set rounds>1 to loop
# the whole list again after a wait, which helps with transient shortages.
import time
from datetime import datetime
from sagemaker.core.resources import Endpoint

INSTANCE_FALLBACK = [
    "ml.g6.xlarge",   # L4 24GB  - model default, usually best availability + cheapest
    "ml.g5.xlarge",   # A10G 24GB - different capacity pool than g5.2xlarge
    "ml.g6.2xlarge",
    "ml.g5.2xlarge",
    "ml.g6.4xlarge",
    "ml.g5.4xlarge",
]

def _is_capacity_error(text):
    t = (text or "").lower()
    return (
        "insufficientinstancecapacity" in t
        or "provision requested ml compute capacity" in t
        # Account-level service quota exhausted for this instance type.
        # Treated the same as capacity: skip to the next instance type.
        or "resourcelimitexceeded" in t
        or "account-level service limit" in t
    )
    
def _cleanup_endpoint(sm, endpoint_name, config_name):
    for fn, kw in (
        (sm.delete_endpoint, {"EndpointName": endpoint_name}),
        (sm.delete_endpoint_config, {"EndpointConfigName": config_name}),
    ):
        try:
            fn(**kw)
        except Exception:
            pass

def deploy_with_fallback(builder, base_name, model_name,
                         instance_types=INSTANCE_FALLBACK,
                         rounds=1, wait_between_rounds=60, poll_seconds=15):
    """Build once, then try deploying across instance_types, falling back on
    capacity errors. Each attempt uses a unique endpoint name so a failed
    attempt never blocks the next. Returns an InService Endpoint, or raises
    RuntimeError if every type is exhausted."""
    sm = boto3.client("sagemaker", region_name=region)
    builder.build(model_name=model_name)
    last = None
    for rnd in range(rounds):
        for it in instance_types:
            stamp = datetime.now().strftime("%Y%m%d%H%M%S")
            ep = f"{base_name}-{it.replace('.', '-')}-{stamp}"
            print(f"[attempt] {it} -> {ep}")
            try:
                builder.deploy(endpoint_name=ep, instance_type=it,
                               endpoint_config_name=ep, model_name=model_name,
                               wait=False)
            except Exception as e:
                last = str(e)
                if _is_capacity_error(last):
                    print(f"  capacity error creating on {it}; trying next. {last[:150]}")
                    _cleanup_endpoint(sm, ep, ep)
                    continue
                raise
            while True:
                status = sm.describe_endpoint(EndpointName=ep)["EndpointStatus"]
                if status in ("InService", "Failed"):
                    break
                time.sleep(poll_seconds)
            if status == "InService":
                print(f"  [ok] InService on {it}: {ep}")
                return Endpoint.get(endpoint_name=ep)
            reason = sm.describe_endpoint(EndpointName=ep).get("FailureReason", "")
            last = reason or status
            print(f"  {status} on {it}: {reason[:150]}")
            _cleanup_endpoint(sm, ep, ep)
            if not _is_capacity_error(reason):
                raise RuntimeError(f"Deploy failed on {it} (non-capacity): {reason}")
        if rnd + 1 < rounds:
            print(f"[round {rnd + 1}] all types short on capacity; waiting {wait_between_rounds}s...")
            time.sleep(wait_between_rounds)
    raise RuntimeError(f"No capacity on any of {instance_types} after {rounds} round(s). Last: {last}")


In [5]:
# Configure the base JumpStart deployment with SDK v3 ModelBuilder, then deploy
# with automatic instance-type fallback on InsufficientInstanceCapacity.
base_model_builder = ModelBuilder.from_jumpstart_config(
    jumpstart_config=jumpstart_config,
    role_arn=role,
    compute=Compute(instance_type=inference_instance_type, instance_count=1),
    sagemaker_session=sess,
)

base_endpoint_requested_name = f"llama-3-2-3b-base-{datetime.now().strftime('%Y%m%d%H%M%S')}"
print("Deploying base model (about 10 minutes once capacity is found)...")
base_model_endpoint = deploy_with_fallback(
    base_model_builder,
    base_name="llama-3-2-3b-base",
    model_name=f"{base_endpoint_requested_name}-model",
)
assert isinstance(base_model_endpoint, Endpoint)


INFO:sagemaker.core.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
Model 'meta-textgeneration-llama-3-2-3b' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-east-1.s3.us-east-1.amazonaws.com/fmhMetadata/eula/llama3_2Eula.txt for terms of use.
INFO:sagemaker.jumpstart:Model 'meta-textgeneration-llama-3-2-3b' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-east-1.s3.us-east-1.amazonaws.com/fmhMetadata/eula/llama3_2Eula.txt for terms of use.
Using model 'meta-textgeneration-llama-3-2-3b' with version '1.1.9'. You can upgrade to version '2.0.0' to g

Deploying base model (about 10 minutes once capacity is found)...


INFO:botocore.tokens:Loading cached SSO token for sso-main
Model 'meta-textgeneration-llama-3-2-3b' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-east-1.s3.us-east-1.amazonaws.com/fmhMetadata/eula/llama3_2Eula.txt for terms of use.
INFO:sagemaker.jumpstart:Model 'meta-textgeneration-llama-3-2-3b' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-east-1.s3.us-east-1.amazonaws.com/fmhMetadata/eula/llama3_2Eula.txt for terms of use.
Using model 'meta-textgeneration-llama-3-2-3b' with wildcard version identifier '*'. You can pin to version '2.0.0' for more stable results. Note that models may have different input/output signatures after a major version upgrade.
DEBUG:sagemaker.core.utils.utils:JumpStart Model ID detected.
DEBUG:sagemaker.core.utils.utils:Either inference spec or model is provided. ModelBuilder is not handling MLflow model input
DEBUG:sagemaker.core.utils.utils:Building for JumpStart model I

[attempt] ml.g6.xlarge -> llama-3-2-3b-base-ml-g6-xlarge-20261028182139


INFO:sagemaker:Creating endpoint with name llama-3-2-3b-base-ml-g6-xlarge-20261028182139
INFO:sagemaker.core.utils.utils:🚀 Deployment started: Endpoint 'llama-3-2-3b-base-ml-g6-xlarge-20261028182139' using DJL_SERVING in SAGEMAKER_ENDPOINT mode (deployment in progress)


  [ok] InService on ml.g6.xlarge: llama-3-2-3b-base-ml-g6-xlarge-20261028182139


In [6]:
from IPython.display import Markdown, display

base_model_endpoint_name = base_model_endpoint.endpoint_name
console_url = (
    f"https://console.aws.amazon.com/sagemaker/home?region={region}"
    f"#/endpoints/{base_model_endpoint_name}"
)
display(Markdown(
    f"You can **[view the base-model endpoint in the SageMaker Console]({console_url})**"
))


You can **[view the base-model endpoint in the SageMaker Console](https://console.aws.amazon.com/sagemaker/home?region=us-east-1#/endpoints/llama-3-2-3b-base-ml-g6-xlarge-20261028182139)**

In [7]:
print(base_model_endpoint_name)
%store base_model_endpoint_name


llama-3-2-3b-base-ml-g6-xlarge-20261028182139
Stored 'base_model_endpoint_name' (str)


## Step 3: Initialize MLflow for Experiment Tracking

### Understanding SageMaker Managed MLflow

SageMaker Serverless MLflow provides a **fully managed experiment MLflow App** that eliminates the need to set up and maintain your own MLflow infrastructure. Key benefits:

- **Centralized Tracking**: All team members log to the same MLflow App
- **No Infrastructure Management**: AWS handles scaling, backups, and availability
- **Integrated Security**: Uses IAM for authentication and authorization
- **Persistent Storage**: Experiments are stored durably in AWS-managed storage

### What Gets Tracked
When you log experiments to MLflow, you capture:
- **Parameters**: Hyperparameters, model IDs, instance types
- **Metrics**: Training loss, validation accuracy, custom metrics
- **Artifacts**: Model files, training datasets, configuration files
- **Metadata**: Run names, timestamps, tags, notes

This creates a **complete audit trail** for governance and compliance.

If you are running this lab as part of an AWS workshop, an MLFlow App has already been created for you. You can use this App to track your fine-tuning experiments. Let's retrieve the MLFlow App URI, you will use it to track the experiments.

If you are running this notebook in your own environment, you need to have an existing running MLFlow App to be able to complete it successfully. Refer to the [AWS Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html) for more details.

Now you are ready to set up your experiment

In [8]:
# Resolve the MLflow App for THIS project. Important: SageMaker Studio often
# pre-sets os.environ["MLFLOW_TRACKING_URI"] to the domain *default* app (e.g.
# smai-ml-std), and the shared loader uses load_dotenv(override=False), so that
# ambient value hides the project value in .env. That is why trusting the env
# var picked the wrong app. So we select by PROJECT_NAME first, then fall back
# to the ARN read straight from the .env FILE, then to any active app.
import os, sys
from pathlib import Path

for _c in [os.getcwd(), *[str(p) for p in Path(os.getcwd()).parents]]:
    if os.path.exists(os.path.join(_c, "workshop_env.py")):
        sys.path.insert(0, _c)
        break
from workshop_env import load_workshop_env, find_env

env = load_workshop_env()
PROJECT_NAME = env["PROJECT_NAME"]

# Read MLFLOW_TRACKING_URI directly from the .env FILE (not os.environ, which
# Studio may have pre-populated with the domain default app).
file_arn = ""
_envp = find_env()
if _envp:
    for _line in _envp.read_text().splitlines():
        _line = _line.strip()
        if _line.startswith("MLFLOW_TRACKING_URI="):
            file_arn = _line.split("=", 1)[1].strip().strip('"').strip("'")
            break

try:
    response = sm_client.list_mlflow_apps(MaxResults=50)
    apps = [a for a in response.get("Summaries", [])
            if a["Status"] in ("Created", "Updated")]
    if not apps:
        raise RuntimeError("No healthy MLflow App found (expected status Created or Updated)")

    by_arn = {a["Arn"]: a for a in apps}
    name_matches = [a for a in apps if PROJECT_NAME in a["Name"]]

    if name_matches:
        chosen, how = name_matches[0], "matched PROJECT_NAME"
    elif file_arn and file_arn in by_arn:
        chosen, how = by_arn[file_arn], "from .env MLFLOW_TRACKING_URI"
    else:
        chosen, how = apps[0], "fallback: first active app"

    mlflow_app_arn = chosen["Arn"]
    mlflow_app_name = chosen["Name"]
    print("\u2713 Using MLflow App for this project:")
    print(f"  Project: {PROJECT_NAME}")
    print(f"  Name:    {mlflow_app_name}")
    print(f"  ARN:     {mlflow_app_arn}")
    print(f"  (selection: {how})")
except Exception as e:
    raise RuntimeError(f"Could not resolve the workshop MLflow App: {e}") from e


✓ Using MLflow App for this project:
  Project: smai-genai-ml-std
  Name:    smai-genai-ml-std-mlflow-app
  ARN:     arn:aws:sagemaker:us-east-1:716664005094:mlflow-app/app-T6L3ELM7AD7B
  (selection: matched PROJECT_NAME)


In [9]:
import mlflow
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

# Connect to the managed MLflow App
mlflow.set_tracking_uri(mlflow_app_arn)

# Create or use existing experiment
# Experiments group related runs together (e.g., all summarization model iterations)
experiment_name = f"summarization-experiment-{timestamp}"
mlflow.set_experiment(experiment_name)

print(f"✓ MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"✓ Experiment: {experiment_name}")

INFO:botocore.tokens:Loading cached SSO token for sso-main
2026/10/28 18:33:11 INFO mlflow.tracking.fluent: Experiment with name 'summarization-experiment-20261028-183305' does not exist. Creating a new experiment.
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main


✓ MLflow tracking URI: arn:aws:sagemaker:us-east-1:716664005094:mlflow-app/app-T6L3ELM7AD7B
✓ Experiment: summarization-experiment-20261028-183305


### 📊 View Your Experiment in MLflow

**To access the MLflow UI:**

1. In the left sidebar of SageMaker Studio, click the **MLflow** icon
2. Click on your MlFlow App name
3. Using the Menu on the right hand side, open MLFlow. (See screenshot below)
4. Navigate to your experiment (See screenshot below. The exact experiment name will differ depending on the timestamp) 
![MLFlow Experiment](../../images/mlflow-console.png)
![MLFlow Experiment](../../images/mlflow-experiment.png)

Next initiate a Run in your experiment to track the fine-tuning job.

In [10]:
# Start a new MLflow run to track this fine-tuning job
# A "run" represents a single training execution with specific parameters
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

run_name=f"llama-3.2-fine-tuning-summarization-{timestamp}"

mlflow_run = mlflow.start_run(run_name=run_name)
print(f"✓ Started MLflow Run ID: {mlflow_run.info.run_id}")
print(f"  This run will track all parameters, metrics, and artifacts from this training job.")

INFO:botocore.tokens:Loading cached SSO token for sso-main


✓ Started MLflow Run ID: 285e3f6149fa49359083375af4d36823
  This run will track all parameters, metrics, and artifacts from this training job.


## Step 4: Load the Prepared Fine-Tuning Dataset from S3

### Data prepared in Lab 2
The fine-tuning dataset is prepared by a standalone notebook: `lab2-data-prep/lab-2a-fine-tuning-data-prep.ipynb`. That notebook acquires a summarization corpus -- either Databricks Dolly 15k filtered to the **summarization** category, or an equivalent corpus generated offline, selected with its `DATA_SOURCE` flag -- runs it through quality gates, splits 70/30 into train/test, creates the instruction prompt `template.json`, and uploads the complete package to S3. Both sources write the same files to the same location, so nothing below changes.

> **Prerequisite**: Run `lab2-data-prep/lab-2a-fine-tuning-data-prep.ipynb` first, using the **same `bucket` value** you set in Step 1. It publishes the dataset to the S3 location referenced below.

### Fine-tuning package (produced by Lab 2)
`s3://<default-bucket>/<ProfileName>/dolly_dataset/`
- `train.jsonl` - training examples
- `test.jsonl` - held-out evaluation examples
- `template.json` - instruction prompt template

Here we reference those S3 paths and pull the files locally for evaluation and inference formatting.

In [11]:
# Resolve the same default-bucket/profile contract produced by Lab 2A.
# Studio user profile name: namespaces this user's S3 prefix.
# Resolution order matters. The .env value comes FIRST because lab0-setup
# discovers it once from the domain, which makes Studio and local kernels agree.
# The old code went straight to the metadata file and otherwise defaulted to
# 'userA', so the same notebook wrote s3://.../Usera/ in Studio and
# s3://.../UserA/ locally -- two datasets, silently.
profile_name = env.get("SAGEMAKER_USER_PROFILE", "")
_profile_source = ".env (SAGEMAKER_USER_PROFILE)"
if not profile_name:
    metadata_path = Path("/opt/ml/metadata/resource-metadata.json")
    if metadata_path.exists():
        profile_name = json.loads(metadata_path.read_text())["UserProfileName"]
        _profile_source = "Studio resource-metadata.json"
    else:
        profile_name = os.environ.get("SM_USER_PROFILE_NAME", "")
        _profile_source = "SM_USER_PROFILE_NAME env var"
if not profile_name:
    raise RuntimeError(
        "Could not resolve the Studio user profile name, which the S3 dataset "
        "prefix is namespaced by. Run lab0-setup/setup.ipynb to populate "
        "SAGEMAKER_USER_PROFILE in .env, or set SM_USER_PROFILE_NAME. Guessing "
        "it would silently read or write the wrong user's prefix."
    )
# Capitalize the first letter to match the existing S3 layout.
profile_name = profile_name[0].upper() + profile_name[1:]

data_location = f"s3://{bucket}/{profile_name}/dolly_dataset"
train_path = "train.jsonl"
template_path = "template.json"
evaluation_path = "test.jsonl"
training_input_path = f"{data_location}/{train_path}"
eval_input_path = f"{data_location}/{evaluation_path}"


def download_s3_uri(uri, local_path=None):
    """Download one S3 object with the AWS client used by SDK v3."""
    bucket_name, key = uri.removeprefix("s3://").split("/", 1)
    destination = local_path or Path(key).name
    s3_client.download_file(bucket_name, key, str(destination))
    return str(destination)


# Local copies are used for MLflow lineage, evaluation, and prompt formatting.
download_s3_uri(training_input_path, train_path)
download_s3_uri(eval_input_path, evaluation_path)
download_s3_uri(f"{data_location}/{template_path}", template_path)

print(f"✓ Referenced fine-tuning dataset at: {data_location}")
print(f"  Training input:   {training_input_path}")
print(f"  Evaluation input: {eval_input_path}")


✓ Referenced fine-tuning dataset at: s3://sagemaker-us-east-1-716664005094/Usera/dolly_dataset
  Training input:   s3://sagemaker-us-east-1-716664005094/Usera/dolly_dataset/train.jsonl
  Evaluation input: s3://sagemaker-us-east-1-716664005094/Usera/dolly_dataset/test.jsonl


In [12]:
from datasets import load_dataset

# ---------------------------------------------------------------------------
# Resolve the prompt template.
# ---------------------------------------------------------------------------
# Two sources, deliberately:
#   * The TRAINING job always uses the S3 template.json in the channel
#     directory. JumpStart reads it from disk inside the container, which has
#     no MLflow client -- that path is fixed and not negotiable.
#   * INFERENCE formatting below prefers the MLflow Prompt Registry, so the run
#     records WHICH prompt version produced these comparisons. Falling back to
#     the downloaded file keeps the lab working when Lab 2A ran without MLflow.
#
# Both must agree. They do by construction: Lab 2A renders template.json FROM
# the registered prompt and asserts the two renderings are identical.
# Shared across users and projects, scoped by use case -- see Lab 2A.
USE_CASE = "text-summarization"
PROMPT_NAME = f"{USE_CASE}-instruct"
PROMPT_ALIAS = "production"

with open(template_path, "r") as f:
    template = json.load(f)          # the S3 file: what training actually used
template_source = f"s3 ({template_path})"
prompt_version_uri = ""

try:
    import mlflow.genai as genai

    # link_to_model=True records the prompt version against this run, so the
    # fine-tuned model carries the template as tracked lineage rather than an
    # untracked file that happened to sit in the channel.
    registry_prompt = genai.load_prompt(
        f"prompts:/{PROMPT_NAME}@{PROMPT_ALIAS}", link_to_model=True
    )
    prompt_version_uri = f"prompts:/{PROMPT_NAME}/{registry_prompt.version}"

    # Trust but verify: the registry prompt must render identically to the
    # template.json the training job consumed. If they diverge, the inference
    # comparison would be measuring a different prompt than the model trained
    # on, which is exactly the silent failure this lab warns about.
    _probe = {"instruction": "Summarize the following text.", "context": "Probe context."}
    if registry_prompt.format(**_probe) == template["prompt"].format(**_probe):
        template_source = f"mlflow ({prompt_version_uri}, verified against S3)"
    else:
        print("⚠ Registry prompt and S3 template.json disagree — using the S3 file,")
        print("  because that is what the training job read. Re-run Lab 2A to resync.")
        prompt_version_uri = ""
except Exception as exc:
    print(f"  (prompt registry unavailable: {type(exc).__name__}: {str(exc)[:120]})")
    print("  Falling back to the S3 template.json.")

# Held-out test split for the base vs. fine-tuned comparison in Step 7.
test_dataset = load_dataset("json", data_files=evaluation_path, split="train")

mlflow.log_param("prompt_source", template_source)
if prompt_version_uri:
    mlflow.log_param("prompt_version_uri", prompt_version_uri)

print(f"✓ Template resolved from {template_source}")
print(f"✓ {test_dataset.num_rows} test examples ready")

INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:httpx2:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"


Generating train split: 0 examples [00:00, ? examples/s]

INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main


✓ Template resolved from mlflow (prompts:/text-summarization-instruct/1, verified against S3)
✓ 360 test examples ready


In [13]:
import pandas as pd

print("Sample training example:")
pd.read_json(train_path, lines=True).iloc[0].to_dict()

Sample training example:


{'instruction': 'Give a concise summary of the passage.',
 'context': 'The Ironworks beat the Kestrels 67 to 61 in a rugby match in Cebu. Concession queues were long at the interval, according to stadium staff. Both sides return to action later in the week. The visiting squad travelled without two of its regular starters. A youth exhibition game preceded the main event. The officiating crew issued no major penalties in the second half. Oscar Delgado led the winners with 4 points in front of a crowd of 61200.',
 'response': 'In Cebu, the Ironworks defeated the Kestrels 67 to 61; Oscar Delgado led with 4 points and attendance was 61200.'}

## Step 5: Log Dataset Lineage to MLflow

The dataset already lives in S3 (uploaded by the Lab 2 data prep notebook). Here we log the training and evaluation datasets as inputs to the active MLflow run so the experiment captures complete data lineage for governance and reproducibility.

In [14]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore', message='Failed to determine whether UCVolumeDatasetSource')

df_train = pd.read_json(train_path, orient="records", lines=True)
training_data = mlflow.data.from_pandas(df_train, source=training_input_path)
mlflow.log_input(training_data, context="training")

INFO:botocore.tokens:Loading cached SSO token for sso-main


In [15]:
df_evaluate = pd.read_json(evaluation_path, orient="records", lines=True)
df_evaluate.size
evaluation_data = mlflow.data.from_pandas(df_evaluate, source=eval_input_path)
mlflow.log_input(evaluation_data, context="evaluation")

INFO:botocore.tokens:Loading cached SSO token for sso-main


## Step 6: Configure and Launch Fine-Tuning with `ModelTrainer`

SDK v3 resolves the JumpStart training image, source code, model artifact input,
default environment, and EULA access from the same `JumpStartConfig` used for
base-model deployment.


<div style="background-color: #d4edda; border: 1px solid #c3e6cb; border-radius: 4px; padding: 12px; margin: 10px 0;">
<b>✓ Llama Model EULA Acceptance</b><br>
To deploy Llama models using SageMaker JumpStart, you must accept Meta's End User License Agreement (EULA). In the notebook, set <code>accept_eula=true</code> in the estimator configuration. By doing so, you acknowledge that you have read and agree to the terms of the EULA, available at https://ai.meta.com/resources/models-and-libraries/llama-downloads/. Deployment will fail if this parameter is not set to true.
</div>


In [16]:
# Fine-tuning configuration. Keep values as strings because JumpStart passes
# them to the training container as hyperparameters.
hyperparameters = {
    "epoch": "2",
    "instruction_tuned": "True",
    "max_input_length": "1024",
}
training_instance_type = "ml.g4dn.12xlarge"  # 4x T4; supported for training and usually far better capacity than the g5 pool
training_output_path = f"s3://{bucket}/{profile_name}/model-output/"

mlflow.log_param("base_model_id", model_id)
mlflow.log_param("model_version", model_version)
for key, value in hyperparameters.items():
    mlflow.log_param(key, value)
print("✓ Hyperparameters configured and logged to MLflow")


INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main


✓ Hyperparameters configured and logged to MLflow


In [17]:
# Configure JumpStart fine-tuning with SDK v3 ModelTrainer.
model_trainer = ModelTrainer.from_jumpstart_config(
    jumpstart_config=jumpstart_config,
    role=role,
    compute=Compute(instance_type=training_instance_type, instance_count=1),
    output_data_config=OutputDataConfig(s3_output_path=training_output_path),
    hyperparameters=hyperparameters,
    sagemaker_session=sess,
    base_job_name="llama-3-2-3b-summarization",
)

mlflow.log_param("instance_type", training_instance_type)
mlflow.log_param("instance_count", 1)
mlflow.log_param("output_path", training_output_path)
mlflow.log_param("training_image_uri", model_trainer.training_image)
print("✓ ModelTrainer configured")
print(f"  Training image: {model_trainer.training_image}")


INFO:sagemaker.core.utils.utils:hub_content_name: meta-textgeneration-llama-3-2-3b, hub_content_version: 1.1.9
INFO:sagemaker.core.utils.utils:Networking not provided. Using default networking:
security_group_ids=None subnets=None enable_network_isolation=True enable_inter_container_traffic_encryption=None
INFO:sagemaker.core.utils.utils:Training image not provided. Using default:
763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04

INFO:sagemaker.core.utils.utils:Using default training dataset: channel_name='training' data_source=S3DataSource(s3_data_type='S3Prefix', s3_uri='s3://jumpstart-cache-prod-us-east-1/training-datasets/sec_amazon/', s3_data_distribution_type='FullyReplicated', attribute_names=None, instance_group_names=Unassigned(), model_access_config=ModelAccessConfig(accept_eula=True), hub_access_config=Unassigned()) content_type=None
INFO:sagemaker.core.utils.utils:Using default model artifact: cha

See https://jumpstart-cache-prod-us-east-1.s3.us-east-1.amazonaws.com/fmhMetadata/eula/llama3_2Eula.txt for terms of use.


DEBUG:sagemaker.core.utils.utils:StoppingCondition not provided. Using default:
max_runtime_in_seconds=3600 max_wait_time_in_seconds=None max_pending_time_in_seconds=None
INFO:sagemaker.core.utils.utils:Training image URI: 763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main


✓ ModelTrainer configured
  Training image: 763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04


### 🔁 Recovery: reattach the completed fine-tuning job
Run the cell below, then **skip** the `model_trainer.train(...)` cell that follows. The job already finished, so this points the freshly configured `model_trainer` at it instead of retraining.

In [18]:
# ===== RECOVERY: run this, then SKIP the model_trainer.train(...) cell below =====
# The fine-tuning job already completed; do NOT retrain. Point the freshly
# configured model_trainer at the finished job so ModelBuilder resolves its
# artifact. This is the exact idiom the SDK recommends.
from sagemaker.core.resources import TrainingJob

#training_job_name = "training-job-name"
#model_artifact_s3 = "s3://model-artifact-location"
#model_trainer._latest_training_job = TrainingJob.get(training_job_name=training_job_name)
#print("Reattached to completed job:")
#rint("  artifact:", model_trainer._latest_training_job.model_artifacts.s3_model_artifacts)


### Start the training job

In [19]:
from IPython.display import Markdown, display

console_url = f"https://console.aws.amazon.com/sagemaker/home?region={region}#/training"

display(Markdown(f"You are now ready to start the training job and fine-tune the model. You can review the metadata and progress of the training job in the AWS console **[🔗 View Training Jobs in SageMaker Console]({console_url})**"))


You are now ready to start the training job and fine-tune the model. You can review the metadata and progress of the training job in the AWS console **[🔗 View Training Jobs in SageMaker Console](https://console.aws.amazon.com/sagemaker/home?region=us-east-1#/training)**

> **⏱️ Note:** The training job will take approximately 15-18 minutes to complete.

In [20]:
# The training channel must point at the DIRECTORY, not only train.jsonl:
# JumpStart instruction tuning also needs template.json beside the training file.
training_data = InputData(channel_name="training", data_source=data_location)

print("🚀 Starting fine-tuning job (approximately 15-18 minutes)...")
model_trainer.train(
    input_data_config=[training_data],
    wait=True,
    logs=True,
)

training_job = model_trainer._latest_training_job
training_job_name = training_job.training_job_name
model_artifact_s3 = training_job.model_artifacts.s3_model_artifacts

print("\n✓ Fine-tuning job completed")
print(f"  Training job:  {training_job_name}")
print(f"  Model artifact: {model_artifact_s3}")
mlflow.log_param("training_job_name", training_job_name)
mlflow.log_param("model_artifact_s3", model_artifact_s3)


🚀 Starting fine-tuning job (approximately 15-18 minutes)...


INFO:sagemaker.core.utils.utils:Training Job Name: llama-3-2-3b-summarization-20261028183336
INFO:sagemaker.config:Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
INFO:sagemaker.config:Not applying SDK defaults from location: /Users/rvvittal/Library/Application Support/sagemaker/config.yaml
INFO:sagemaker.core.resources:Creating training_job resource.
DEBUG:sagemaker.core.utils.utils:No config provided. Using default config.


Output()

INFO:botocore.tokens:Loading cached SSO token for sso-main


INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
Starting training script
INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
++ /opt/conda/bin/python3 --version
INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
Python 3.10.8
INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
++ echo /opt/ml/input/config/resourceconfig.json:
INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
/opt/ml/input/config/resourceconfig.json:
INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
++ cat /opt/ml/input/config/resourceconfig.json
INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
++ echo
INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
++ echo /opt/ml/input/config/inputdataconfig.json

...

INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
INFO:root:Combining pre-trained base model with the PEFT adapter module.
INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]#015Loading checkpoint shards:  50%|█████     | 1/2 [00:00<00:00,  8.45it/s]#015Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 10.19it/s]
INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
INFO:root:Saving the combined model in safetensors format.
INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
INFO:root:Saving complete.
INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
INFO:root:Copying tokenizer to the output directory.
INFO:sagemaker.core.resources:llama-3-2-3b-summarization-20261028183336/algo-1-1793237965:
INFO:root:Putting inferen

INFO:sagemaker.core.resources:Final Resource Status: [bold]Completed



✓ Fine-tuning job completed
  Training job:  llama-3-2-3b-summarization-20261028183336
  Model artifact: s3://sagemaker-us-east-1-716664005094/Usera/model-output/llama-3-2-3b-summarization-20261028183336/output/model


INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main


's3://sagemaker-us-east-1-716664005094/Usera/model-output/llama-3-2-3b-summarization-20261028183336/output/model'

###  Track the training progress
While waiting, you can track the training progress above and also review the information you have logged in MLFLow:
1. Navigate to the MLFlow console
2. Find the summarization - experiment you created earlier
3. Click on its name to view the experiment details
4. Locate the Run and click on its name to view its details

![MLFlow Experiment](../../images/run.png)
![MLFlow Experiment](../../images/run_details.png)

In [21]:
# Record training output immediately; Step 7 adds the inference image and
# environment after ModelBuilder resolves them during deployment.
mlflow.log_dict(
    {
        "model_artifact": model_artifact_s3,
        "training_job_name": training_job_name,
        "base_model_id": model_id,
        "base_model_version": model_version,
    },
    "training_info.json",
)


INFO:botocore.tokens:Loading cached SSO token for sso-main


In [22]:
from IPython.display import Markdown, display

s3_path = model_artifact_s3.removeprefix("s3://").split("/", 1)
console_url = f"https://s3.console.aws.amazon.com/s3/object/{s3_path[0]}?prefix={s3_path[1]}"
display(Markdown(
    f"**Training output:** [{model_artifact_s3}]({console_url})"
))


**Training output:** [s3://sagemaker-us-east-1-716664005094/Usera/model-output/llama-3-2-3b-summarization-20261028183336/output/model](https://s3.console.aws.amazon.com/s3/object/sagemaker-us-east-1-716664005094?prefix=Usera/model-output/llama-3-2-3b-summarization-20261028183336/output/model)

## Step 7: Deploy the Fine-Tuned Model with `ModelBuilder`


> **⏱️ Note:** The deployment job will take approximately 10 minutes to complete.

In [23]:
# ModelBuilder recognizes a JumpStart ModelTrainer, takes the completed job's
# model artifact, and resolves the matching JumpStart inference image. Deploy
# with the same capacity-aware fallback across instance types.
fine_tuned_endpoint_requested_name = (
    f"llama-3-2-3b-finetuned-{datetime.now().strftime('%Y%m%d%H%M%S')}"
)
fine_tuned_model_builder = ModelBuilder(
    model=model_trainer,
    role_arn=role,
    compute=Compute(instance_type=inference_instance_type, instance_count=1),
    env_vars={"OPTION_MAX_MODEL_LEN": "65536"},
    sagemaker_session=sess,
)

print("Deploying fine-tuned model with instance-type fallback...")
fine_tuned_model_endpoint = deploy_with_fallback(
    fine_tuned_model_builder,
    base_name="llama-3-2-3b-finetuned",
    model_name=f"{fine_tuned_endpoint_requested_name}-model",
)
assert isinstance(fine_tuned_model_endpoint, Endpoint)


DEBUG:sagemaker.core.utils.utils:Either inference spec or model is provided. ModelBuilder is not handling MLflow model input


Deploying fine-tuned model with instance-type fallback...


Model 'meta-textgeneration-llama-3-2-3b' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-east-1.s3.us-east-1.amazonaws.com/fmhMetadata/eula/llama3_2Eula.txt for terms of use.
INFO:sagemaker.jumpstart:Model 'meta-textgeneration-llama-3-2-3b' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-east-1.s3.us-east-1.amazonaws.com/fmhMetadata/eula/llama3_2Eula.txt for terms of use.
Using model 'meta-textgeneration-llama-3-2-3b' with wildcard version identifier '*'. You can pin to version '2.0.0' for more stable results. Note that models may have different input/output signatures after a major version upgrade.
DEBUG:sagemaker.core.utils.utils:JumpStart Model ID detected.
DEBUG:sagemaker.core.utils.utils:Building for JumpStart model ID...
Model 'meta-textgeneration-llama-3-2-3b' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-east-1.s3.us-east-1.amazonaws.com/fmhMetadata/eu

[attempt] ml.g6.xlarge -> llama-3-2-3b-finetuned-ml-g6-xlarge-20261028185919


INFO:sagemaker:Creating endpoint with name llama-3-2-3b-finetuned-ml-g6-xlarge-20261028185919
INFO:sagemaker.core.utils.utils:🚀 Deployment started: Endpoint 'llama-3-2-3b-finetuned-ml-g6-xlarge-20261028185919' using DJL_SERVING in SAGEMAKER_ENDPOINT mode (deployment in progress)


  [ok] InService on ml.g6.xlarge: llama-3-2-3b-finetuned-ml-g6-xlarge-20261028185919


In [24]:
fine_tuned_model_endpoint_name = fine_tuned_model_endpoint.endpoint_name
print(fine_tuned_model_endpoint_name)
%store fine_tuned_model_endpoint_name


llama-3-2-3b-finetuned-ml-g6-xlarge-20261028185919
Stored 'fine_tuned_model_endpoint_name' (str)


In [25]:
# After build/deploy, ModelBuilder holds the resolved inference image and the
# complete JumpStart serving environment. Lab 3C registers exactly these values,
# rather than incorrectly registering the training image.
inference_image_uri = fine_tuned_model_builder.image_uri
inference_environment = dict(fine_tuned_model_builder.env_vars)

assert inference_image_uri, "ModelBuilder did not resolve an inference image"
mlflow.log_param("base_endpoint_name", base_model_endpoint_name)
mlflow.log_param("endpoint_name", fine_tuned_model_endpoint_name)
mlflow.log_param("endpoint_instance_type", inference_instance_type)
mlflow.log_param("inference_image_uri", inference_image_uri)
mlflow.log_dict(
    {
        "model_artifact": model_artifact_s3,
        "training_job_name": training_job_name,
        "inference_image_uri": inference_image_uri,
        "inference_environment": inference_environment,
        "base_model_id": model_id,
        "base_model_version": model_version,
    },
    "model_info.json",
)
print(f"✓ Inference image recorded for Lab 3C: {inference_image_uri}")


INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:SSO Token refresh succeeded
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main


✓ Inference image recorded for Lab 3C: 763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.29.0-lmi11.0.0-cu124


## Step 8: Compare the Base and Fine-Tuned Models


Let's now do some initial testing to compare the outputs of the base and fine-tuned models

In [26]:
def invoke_json(endpoint, payload, *, accept_eula=False):
    """Invoke a typed SDK v3 Endpoint and decode its JSON response."""
    output = endpoint.invoke(
        body=json.dumps(payload),
        content_type="application/json",
        accept="application/json",
        custom_attributes="accept_eula=true" if accept_eula else None,
    )
    body = output.body.read() if hasattr(output.body, "read") else output.body
    if isinstance(body, bytes):
        body = body.decode("utf-8")
    return json.loads(body) if isinstance(body, str) else body


def generated_text(response):
    """Normalize common text-generation response envelopes."""
    if isinstance(response, list) and len(response) == 1:
        response = response[0]
    if not isinstance(response, dict) or "generated_text" not in response:
        raise ValueError(f"Unexpected endpoint response shape: {response!r}")
    return response["generated_text"]


def print_response(model_label, payload, response):
    print(f"Model: {model_label}")
    print(f"Prompt: {payload['inputs']}")
    print(f"Response: {generated_text(response)}")
    print("\n==================================\n")


payload = {
    "inputs": "### Instruction: What is Amazon SageMaker in one sentence?### Response:\n",
    "parameters": {
        "max_new_tokens": 128,
        "top_p": 0.9,
        "temperature": 0.6,
        "return_full_text": False,
    },
}

response = invoke_json(fine_tuned_model_endpoint, payload)
print_response(f"{model_id} (fine-tuned)", payload, response)


Model: meta-textgeneration-llama-3-2-3b (fine-tuned)
Prompt: ### Instruction: What is Amazon SageMaker in one sentence?### Response:

Response: Amazon SageMaker is a fully managed service that enables developers and data scientists to quickly and easily build, train, and deploy machine learning models at any scale. With Amazon SageMaker, you can use a variety of pre-built algorithms or train your own models using your own data. You can also use Amazon SageMaker to create custom algorithms and train them on your own data. Amazon SageMaker makes it easy to deploy your models in production, so you can start generating insights from your data right away.




In [27]:
import pandas as pd
from IPython.display import display, HTML

inputs, ground_truth_responses = [], []
responses_before_finetuning, responses_after_finetuning = [], []


def predict_and_print(datapoint):
    input_output_demarkation_key = "\n\n### Response:\n"
    payload = {
        "inputs": template["prompt"].format(
            instruction=datapoint["instruction"], context=datapoint["context"]
        ) + input_output_demarkation_key,
        "parameters": {"max_new_tokens": 100},
    }
    inputs.append(payload["inputs"])
    ground_truth_responses.append(datapoint["response"])

    pretrained_response = invoke_json(base_model_endpoint, payload, accept_eula=True)
    responses_before_finetuning.append(generated_text(pretrained_response))

    finetuned_response = invoke_json(fine_tuned_model_endpoint, payload)
    responses_after_finetuning.append(generated_text(finetuned_response))


for datapoint in test_dataset.select(range(min(5, test_dataset.num_rows))):
    predict_and_print(datapoint)

df = pd.DataFrame({
    "Inputs": inputs,
    "Ground Truth": ground_truth_responses,
    "Response from non-finetuned model": responses_before_finetuning,
    "Response from fine-tuned model": responses_after_finetuning,
})
display(HTML(df.to_html()))


,Inputs,Ground Truth,Response from non-finetuned model,Response from fine-tuned model
0,"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nProvide a brief summary of this report.\n\n### Input:\nAt a September trade show in Fukuoka, Vantage Textiles unveiled the Meridian sensor array. Engineers say the design cuts energy use by 9 percent compared with the equipment most renewable energy operators run today. List price is 32400 dollars, with 24750 units scheduled for the first year of production. Several competing exhibitors used the floor to show refreshed versions of older product lines. Catering staff reported that the coffee stands were busiest in the mid-morning slot. Badges were scanned at the door, a change from the paper passes used previously. A panel discussion on workforce training ran alongside the exhibition hall.\n\n\n\n### Response:\n","Vantage Textiles launched the Meridian sensor array in Fukuoka in September, priced at 32400 dollars with 24750 first-year units planned, and claims 9 percent lower energy use than typical renewable energy equipment.","The Meridian sensor array is a new product that can reduce energy use by 9 percent. The list price is 32400 dollars, and 24750 units are scheduled for the first year of production. Several competing exhibitors showed refreshed versions of older product lines, and the coffee stands were busiest in the mid-morning slot. Badges were scanned at the door, a change from the paper passes used previously. A panel discussion on workforce training ran alongside the exhibition hall.","The Meridian sensor array is a new product that can reduce energy use by 9 percent. The list price is 32400 dollars, and 24750 units are scheduled for the first year of production. Several competing exhibitors showed refreshed versions of older product lines, and the coffee stands were busiest in the mid-morning slot. Badges were scanned at the door, a change from the paper passes used previously. A panel discussion on workforce training ran alongside the exhibition hall."
1,"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nCondense the article into one or two sentences.\n\n### Input:\nA flood in Adelaide brought 222 millimetres of rain over 22 hours in March. 10150 residents were relocated to shelters and 111 schools shut their doors. Forecasters had issued advisories earlier in the week. A community centre opened as a daytime warming and charging station. Insurers expect claim volumes to be clearer in a few weeks. A text alert system was used for the first time since it was installed. Mei-Lin Zhao, the emergency coordinator, reported main routes reopened inside 24 hours.\n\n\n\n### Response:\n","A March flood dropped 222 millimetres of rain on Adelaide in 22 hours, prompting 10150 evacuations and 111 school closures; emergency coordinator Mei-Lin Zhao said main routes reopened within a day.","A flood in Adelaide brought 222 millimetres of rain over 22 hours in March. 10150 residents were relocated to shelters and 111 schools shut their doors. Forecasters had issued advisories earlier in the week. A community centre opened as a daytime warming and charging station. Insurers expect claim volumes to be clearer in a few weeks. A text alert system was used for the first time since it was installed. Mei-Lin Zhao, the emergency coordinator, reported main routes reopened inside","A flood in Adelaide brought 222 millimetres of rain over 22 hours in March. 10150 residents were relocated to shelters and 111 schools shut their doors. Forecasters had issued advisories earlier in the week. A community centre opened as a daytime warming and charging station. Insurers expect claim volumes to be clearer in a few weeks. A text alert system was used for the fi

In [28]:
print(f"Experiment: {experiment_name}")
print(f"Run ID:     {mlflow_run.info.run_id}")


Experiment: summarization-experiment-20261028-183305
Run ID:     285e3f6149fa49359083375af4d36823


In [29]:
# Persist the exact handoff consumed by Lab 3C.
run_id = mlflow_run.info.run_id
%store experiment_name
%store run_id
%store training_job_name
%store base_model_endpoint_name
%store fine_tuned_model_endpoint_name


Stored 'experiment_name' (str)
Stored 'run_id' (str)
Stored 'training_job_name' (str)
Stored 'base_model_endpoint_name' (str)
Stored 'fine_tuned_model_endpoint_name' (str)


## Step 9: Review Governance Artifacts in MLflow

### What We've Tracked

Throughout this lab, we've automatically logged:

1. **Data Lineage**
   - Source dataset location
   - Number of training/test examples
   - Data preprocessing steps

2. **Model Lineage**
   - Base model ID and version
   - All hyperparameters
   - Training job name
   - Model artifact location

3. **Deployment Lineage**
   - Endpoint container image
   - Instance type

### Accessing Your Experiments

You can view all tracked experiments in:
1. **SageMaker Studio**: Navigate to MLflow App
2. **MLflow UI**: Access through the MLflow App URL
3. **Programmatically**: Query using MLflow APIs

Since the training is now complete, let's mark the Run as completed.

In [30]:
mlflow.end_run()

print("✓ MLflow run completed")
print("\nRun Summary:")
print(f"  Experiment:       {experiment_name}")
print(f"  Run ID:           {run_id}")
print(f"  Training job:     {training_job_name}")
print(f"  Model artifact:   {model_artifact_s3}")
print(f"  Inference image:  {inference_image_uri}")
print("\nAll parameters, metrics, artifacts, and deployment lineage are recorded.")


INFO:botocore.tokens:Loading cached SSO token for sso-main
INFO:botocore.tokens:Loading cached SSO token for sso-main


🏃 View run llama-3.2-fine-tuning-summarization-20261028-183313 at: https://mlflow.sagemaker.us-east-1.app.aws/#/experiments/3/runs/285e3f6149fa49359083375af4d36823
🧪 View experiment at: https://mlflow.sagemaker.us-east-1.app.aws/#/experiments/3
✓ MLflow run completed

Run Summary:
  Experiment:       summarization-experiment-20261028-183305
  Run ID:           285e3f6149fa49359083375af4d36823
  Training job:     llama-3-2-3b-summarization-20261028183336
  Model artifact:   s3://sagemaker-us-east-1-716664005094/Usera/model-output/llama-3-2-3b-summarization-20261028183336/output/model
  Inference image:  763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.29.0-lmi11.0.0-cu124

All parameters, metrics, artifacts, and deployment lineage are recorded.


## Key Takeaways

In this lab, you learned how to:

1. ✅ **Fine-tune a foundation model** for a specific use case (text summarization)
2. ✅ **Track all experimentation** using SageMaker Managed MLflow
3. ✅ **Establish complete lineage** from data → training → deployment
4. ✅ **Create audit trails** for governance and compliance
5. ✅ **Enable reproducibility** by logging all parameters and artifacts

### Governance Benefits Demonstrated

- **Auditability**: Every training run is logged with complete metadata
- **Reproducibility**: Any experiment can be recreated from tracked parameters
- **Lineage**: Clear chain from source data to deployed model
- **Compliance**: Meet regulatory requirements for model documentation
- **Collaboration**: Team members can view and compare all experiments

### Next Steps

- In the next lab, you will evaluate your fine-tuned model, review the metrics, and register it to the SageMaker Model Registry.